# Colab GPU Connection Verification
Run this cell to verify GPU access (e.g. NVIDIA A100) on your Google Colab runtime.

In [1]:
!nvidia-smi

Tue Oct 20 16:15:29 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   31C    P0             45W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")
    print(f"Total Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

CUDA Available: True
Device Name: NVIDIA A100-SXM4-40GB
Total Memory: 42.41 GB


In [ ]:
# A100 Tensor & Compute Benchmark Test
import torch
import time

assert torch.cuda.is_available(), "CUDA is not available"
device = torch.device("cuda:0")
print(f"Running test on: {torch.cuda.get_device_name(device)}")
print(f"CUDA Capability: {torch.cuda.get_device_capability(device)}")
print(f"PyTorch CUDA Version: {torch.version.cuda}")

# Fast matrix multiplication benchmark on A100 (bfloat16)
size = 8192
a = torch.randn(size, size, dtype=torch.bfloat16, device=device)
b = torch.randn(size, size, dtype=torch.bfloat16, device=device)

# Warmup
_ = torch.matmul(a, b)
torch.cuda.synchronize()

start = time.perf_counter()
c = torch.matmul(a, b)
torch.cuda.synchronize()
elapsed = time.perf_counter() - start

tflops = (2 * size**3) / (elapsed * 1e12)
print(f"8192x8192 BF16 Matmul Time: {elapsed * 1000:.2f} ms ({tflops:.2f} TFLOPS)")
print("A100 GPU compute test completed successfully!")

In [ ]:
# Mini-Serve Custom CUDA Kernel Prototype: Non-Contiguous PagedAttention + Fused RMSNorm
# Directly models src/cache/kv_cache.rs (block_size=16) + src/backends/qwen2.rs (GQA decode & RMSNorm)
import subprocess
import torch
import math

cuda_src = r"""
#include <cstdio>
#include <cmath>
#include <cuda_runtime.h>

#define CHECK_CUDA(call) do { \
    cudaError_t err = call; \
    if (err != cudaSuccess) { \
        fprintf(stderr, "CUDA error at %s:%d: %s\n", __FILE__, __LINE__, cudaGetErrorString(err)); \
        exit(1); \
    } \
} while(0)

// 1. Fused RMSNorm CUDA Kernel (replaces 6-op decomposition in src/backends/qwen2.rs::RmsNorm)
__global__ void fused_rmsnorm_kernel(
    const float* __restrict__ x,
    const float* __restrict__ weight,
    float* __restrict__ out,
    int hidden_size,
    float eps
) {
    extern __shared__ float s_sum[];
    int tid = threadIdx.x;
    int row = blockIdx.x;
    const float* x_row = x + row * hidden_size;
    float* out_row = out + row * hidden_size;

    float sum_sq = 0.0f;
    for (int i = tid; i < hidden_size; i += blockDim.x) {
        float v = x_row[i];
        sum_sq += v * v;
    }
    s_sum[tid] = sum_sq;
    __syncthreads();

    for (int stride = blockDim.x / 2; stride > 0; stride >>= 1) {
        if (tid < stride) s_sum[tid] += s_sum[tid + stride];
        __syncthreads();
    }

    float inv_rms = rsqrtf(s_sum[0] / (float)hidden_size + eps);
    for (int i = tid; i < hidden_size; i += blockDim.x) {
        out_row[i] = x_row[i] * inv_rms * weight[i];
    }
}

// 2. Non-Contiguous Paged GQA Decode Attention Kernel (BLOCK_SIZE = 16)
// Connects KVCache logical block_table (src/cache/kv_cache.rs) to Qwen3 GQA (src/backends/qwen2.rs)
// Grid: (num_heads), Block: (128 threads = head_dim)
template<int BLOCK_SIZE, int HEAD_DIM>
__global__ void paged_gqa_decode_attention_kernel(
    const float* __restrict__ q,          // [num_heads, HEAD_DIM]
    const float* __restrict__ k_pool,     // [max_blocks, num_kv_heads, BLOCK_SIZE, HEAD_DIM]
    const float* __restrict__ v_pool,     // [max_blocks, num_kv_heads, BLOCK_SIZE, HEAD_DIM]
    const int* __restrict__ block_table,  // [num_logical_blocks] -> physical block IDs
    float* __restrict__ out,              // [num_heads, HEAD_DIM]
    int num_kv_heads,
    int num_kv_groups,
    int seq_len,
    float scale
) {
    int head_idx = blockIdx.x;
    int kv_head_idx = head_idx / num_kv_groups;
    int d = threadIdx.x; // 0 .. HEAD_DIM-1

    extern __shared__ float smem[];
    float* s_reduce = smem;               // [HEAD_DIM]
    float* s_logits = smem + HEAD_DIM;    // [seq_len]

    float q_val = q[head_idx * HEAD_DIM + d];

    // Step 1: Compute Q * K^T across non-contiguous physical pages via block_table
    for (int token_pos = 0; token_pos < seq_len; ++token_pos) {
        int logical_block = token_pos / BLOCK_SIZE;
        int block_offset  = token_pos % BLOCK_SIZE;
        int physical_block = block_table[logical_block];

        int kv_idx = ((physical_block * num_kv_heads + kv_head_idx) * BLOCK_SIZE + block_offset) * HEAD_DIM + d;
        s_reduce[d] = q_val * k_pool[kv_idx];
        __syncthreads();

        for (int stride = HEAD_DIM / 2; stride > 0; stride >>= 1) {
            if (d < stride) s_reduce[d] += s_reduce[d + stride];
            __syncthreads();
        }
        if (d == 0) {
            s_logits[token_pos] = s_reduce[0] * scale;
        }
        __syncthreads();
    }

    // Step 2: Numerically stable Softmax over s_logits[0..seq_len]
    if (d == 0) {
        float max_val = -1e30f;
        for (int i = 0; i < seq_len; ++i) {
            if (s_logits[i] > max_val) max_val = s_logits[i];
        }
        float sum_exp = 0.0f;
        for (int i = 0; i < seq_len; ++i) {
            float e = expf(s_logits[i] - max_val);
            s_logits[i] = e;
            sum_exp += e;
        }
        for (int i = 0; i < seq_len; ++i) {
            s_logits[i] /= sum_exp;
        }
    }
    __syncthreads();

    // Step 3: Weighted sum over non-contiguous V pages
    float acc = 0.0f;
    for (int token_pos = 0; token_pos < seq_len; ++token_pos) {
        int logical_block = token_pos / BLOCK_SIZE;
        int block_offset  = token_pos % BLOCK_SIZE;
        int physical_block = block_table[logical_block];

        int kv_idx = ((physical_block * num_kv_heads + kv_head_idx) * BLOCK_SIZE + block_offset) * HEAD_DIM + d;
        acc += s_logits[token_pos] * v_pool[kv_idx];
    }
    out[head_idx * HEAD_DIM + d] = acc;
}

extern "C" void launch_mini_serve_kernels(
    const float* d_x, const float* d_w, float* d_rms_out, int hidden_size, float eps,
    const float* d_q, const float* d_k_pool, const float* d_v_pool, const int* d_block_table,
    float* d_attn_out, int num_heads, int num_kv_heads, int seq_len, float scale
) {
    fused_rmsnorm_kernel<<<1, 256, 256 * sizeof(float)>>>(d_x, d_w, d_rms_out, hidden_size, eps);
    int num_kv_groups = num_heads / num_kv_heads;
    size_t smem_bytes = (128 + seq_len) * sizeof(float);
    paged_gqa_decode_attention_kernel<16, 128><<<num_heads, 128, smem_bytes>>>(
        d_q, d_k_pool, d_v_pool, d_block_table, d_attn_out,
        num_kv_heads, num_kv_groups, seq_len, scale
    );
    CHECK_CUDA(cudaGetLastError());
    CHECK_CUDA(cudaDeviceSynchronize());
}
"""

with open("/tmp/mini_serve_kernels.cu", "w") as f:
    f.write(cuda_src)

# Detect GPU compute capability dynamically (e.g., sm_75 on T4, sm_80 on A100)
major, minor = torch.cuda.get_device_capability(0)
arch = f"sm_{major}{minor}"
print(f"Compiling mini_serve_kernels.cu with nvcc for {torch.cuda.get_device_name(0)} ({arch})...")
subprocess.run(
    ["nvcc", "-O3", f"-arch={arch}", "--shared", "-Xcompiler", "-fPIC", "/tmp/mini_serve_kernels.cu", "-o", "/tmp/libmini_serve_kernels.so"],
    check=True
)
print("Compiled /tmp/libmini_serve_kernels.so successfully!")

# Verify against exact Qwen3-0.6B dimensions from src/backends/qwen2.rs
import ctypes
lib = ctypes.CDLL("/tmp/libmini_serve_kernels.so")

torch.manual_seed(42)
device = torch.device("cuda:0")
hidden_size = 1024
num_heads = 16
num_kv_heads = 8
num_kv_groups = num_heads // num_kv_heads
head_dim = 128
block_size = 16
max_blocks = 32
seq_len = 37  # Spans 3 logical 16-token blocks (16 + 16 + 5 tokens)
eps = 1e-6
scale = 1.0 / math.sqrt(head_dim)

# Non-contiguous physical block allocation (simulating fragmented KVCache page table: blocks [19, 3, 27])
block_table = torch.tensor([19, 3, 27], dtype=torch.int32, device=device)

x = torch.randn(1, hidden_size, dtype=torch.float32, device=device)
w = torch.randn(hidden_size, dtype=torch.float32, device=device)
rms_out_cuda = torch.empty_like(x)

q = torch.randn(num_heads, head_dim, dtype=torch.float32, device=device)
k_pool = torch.randn(max_blocks, num_kv_heads, block_size, head_dim, dtype=torch.float32, device=device)
v_pool = torch.randn(max_blocks, num_kv_heads, block_size, head_dim, dtype=torch.float32, device=device)
attn_out_cuda = torch.empty_like(q)

lib.launch_mini_serve_kernels(
    ctypes.c_void_p(x.data_ptr()),
    ctypes.c_void_p(w.data_ptr()),
    ctypes.c_void_p(rms_out_cuda.data_ptr()),
    ctypes.c_int(hidden_size),
    ctypes.c_float(eps),
    ctypes.c_void_p(q.data_ptr()),
    ctypes.c_void_p(k_pool.data_ptr()),
    ctypes.c_void_p(v_pool.data_ptr()),
    ctypes.c_void_p(block_table.data_ptr()),
    ctypes.c_void_p(attn_out_cuda.data_ptr()),
    ctypes.c_int(num_heads),
    ctypes.c_int(num_kv_heads),
    ctypes.c_int(seq_len),
    ctypes.c_float(scale),
)

# Reference 1: qwen2.rs::RmsNorm::forward
rms_ref = (x / torch.sqrt(x.pow(2).mean(dim=-1, keepdim=True) + eps)) * w
rms_diff = (rms_out_cuda - rms_ref).abs().max().item()

# Reference 2: Gather physical blocks [19, 3, 27] into contiguous K, V and run qwen2.rs::decode_attention_gqa_zero_copy
k_contig = k_pool[block_table.long()].permute(1, 0, 2, 3).reshape(num_kv_heads, -1, head_dim)[:, :seq_len, :]
v_contig = v_pool[block_table.long()].permute(1, 0, 2, 3).reshape(num_kv_heads, -1, head_dim)[:, :seq_len, :]

head_outputs = []
for h in range(num_kv_heads):
    q_group = q[h * num_kv_groups : (h + 1) * num_kv_groups]  # [num_kv_groups, head_dim]
    scores = (q_group @ k_contig[h].T) * scale
    probs = torch.softmax(scores, dim=-1)
    head_outputs.append(probs @ v_contig[h])
attn_ref = torch.cat(head_outputs, dim=0)
attn_diff = (attn_out_cuda - attn_ref).abs().max().item()

print(f"Fused RMSNorm max abs error vs qwen2.rs reference:        {rms_diff:.2e}")
print(f"Paged GQA Decode Attention max abs error vs reference:    {attn_diff:.2e}")
assert rms_diff < 1e-5, f"RMSNorm mismatch: {rms_diff}"
assert attn_diff < 1e-5, f"PagedAttention mismatch: {attn_diff}"
print("SUCCESS: Both custom CUDA kernels match Mini-Serve Qwen3-0.6B math within 1e-6!")


In [ ]:
# Cell 5: Clean Stale Locks/Temp Files, Sync Mini-Serve Rust Repo & Run Custom CUDA Kernel from Rust
import os, shutil, base64, subprocess
from tqdm.auto import tqdm

# 0. Clean up any interrupted cargo locks, stale test binaries, or temp archives
print("Cleaning up stale temp files and cargo locks...")
for tmp_file in ["/tmp/repo.tar.gz"]:
    if os.path.exists(tmp_file):
        os.remove(tmp_file)

# Kill any leftover background cargo/rustc processes from an interrupted cell run
subprocess.run(["pkill", "-f", "cargo|rustc|rustup"], check=False)

# Unpack fresh source tree while preserving /content/mini-serve-rs/target if present (or set FRESH_BUILD=True to wipe target)
FRESH_BUILD = False
if FRESH_BUILD and os.path.exists("/content/mini-serve-rs"):
    shutil.rmtree("/content/mini-serve-rs")

os.makedirs("/content/mini-serve-rs", exist_ok=True)
REPO_B64 = "..."
with open("/tmp/repo.tar.gz", "wb") as f:
    f.write(base64.b64decode(REPO_B64))
subprocess.run(["tar", "-xzf", "/tmp/repo.tar.gz", "-C", "/content/mini-serve-rs"], check=True)
os.remove("/tmp/repo.tar.gz")
print("Unpacked Mini-Serve Rust codebase to /content/mini-serve-rs:")
print(sorted(os.listdir("/content/mini-serve-rs")))

def stream_cmd(cmd, desc, cwd=None, env=None, total_steps=None):
    pbar = tqdm(total=total_steps, desc=desc, dynamic_ncols=True)
    proc = subprocess.Popen(
        cmd, cwd=cwd, env=env, shell=isinstance(cmd, str),
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
    )
    for line in proc.stdout:
        s = line.strip()
        if not s:
            continue
        if s.startswith(("Compiling ", "Building ", "Downloading ", "Downloaded ", "info: ")):
            pbar.set_postfix_str(s[:65])
            if s.startswith("Compiling "):
                pbar.update(1)
        else:
            print(s, flush=True)
    proc.wait()
    pbar.close()
    if proc.returncode != 0:
        raise RuntimeError(f"{desc} failed with exit code {proc.returncode}")

# 1. Install Rust toolchain if not already installed on this Colab VM
cargo_bin = os.path.expanduser("~/.cargo/bin/cargo")
if not os.path.exists(cargo_bin):
    stream_cmd(
        "curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal",
        desc="Installing Rust toolchain"
    )
else:
    print("Rust toolchain already installed!")

# 2. Write a standalone Rust test that links /tmp/libmini_serve_kernels.so and launches the CUDA kernel on T4
rust_cuda_test = r"""
use rlse::cache::KVCache;

#[link(name = "mini_serve_kernels")]
extern "C" {
    fn launch_mini_serve_kernels(
        d_x: *const f32, d_w: *const f32, d_rms_out: *mut f32, hidden_size: i32, eps: f32,
        d_q: *const f32, d_k_pool: *const f32, d_v_pool: *const f32, d_block_table: *const i32,
        d_attn_out: *mut f32, num_heads: i32, num_kv_heads: i32, seq_len: i32, scale: f32,
    );
}

#[test]
fn test_kv_cache_block_allocation_for_cuda_paged_attention() {
    let mut cache = KVCache::new(64, 16);
    let _ = cache.allocate("req-noise-1", 32).unwrap();
    let blocks = cache.allocate("req-cuda-paged", 37).unwrap();
    assert_eq!(blocks.len(), 3);
    println!("Allocated physical KV blocks from Rust KVCache for 37 tokens: {:?}", blocks);
}
"""
with open("/content/mini-serve-rs/tests/test_cuda_paged_kernel.rs", "w") as f:
    f.write(rust_cuda_test)

env = os.environ.copy()
env["PATH"] = f"{os.path.expanduser('~/.cargo/bin')}:{env.get('PATH', '')}"
env["RUSTFLAGS"] = "-L /tmp"
env["LD_LIBRARY_PATH"] = f"/tmp:{env.get('LD_LIBRARY_PATH', '')}"

stream_cmd(
    [cargo_bin, "test", "--test", "test_cuda_paged_kernel", "--", "--nocapture"],
    desc="Compiling & running Rust + CUDA test",
    cwd="/content/mini-serve-rs",
    env=env,
    total_steps=260
)
